In [2]:
## SETUP
import pandas as pd
from sqlalchemy import create_engine, text
import sys, os

sys.path.insert(0, os.path.abspath('..'))
import config

engine = create_engine(
    f"mysql+pymysql://{config.DB_CONFIG['user']}:{config.DB_CONFIG['password']}"
    f"@{config.DB_CONFIG['host']}:{config.DB_CONFIG['port']}/{config.DB_CONFIG['database']}"
)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_rows', 20)
print("Ready.")

Ready.


In [3]:
## Q1: TOTAL RETURN PER STOCK (FULL PERIOD)
q1 = """
WITH first_date AS (SELECT ticker, MIN(trade_date) AS dt FROM daily_prices GROUP BY ticker),
     last_date  AS (SELECT ticker, MAX(trade_date) AS dt FROM daily_prices GROUP BY ticker)
SELECT s.ticker, s.company_name, s.market, s.sector,
       fp.close_price AS start_price, lp.close_price AS end_price,
       ROUND(((lp.close_price - fp.close_price)/fp.close_price)*100,2) AS total_return_pct
FROM stocks s
JOIN first_date fd ON s.ticker = fd.ticker
JOIN daily_prices fp ON s.ticker = fp.ticker AND fp.trade_date = fd.dt
JOIN last_date ld ON s.ticker = ld.ticker
JOIN daily_prices lp ON s.ticker = lp.ticker AND lp.trade_date = ld.dt
ORDER BY total_return_pct DESC
"""
df_q1 = pd.read_sql(q1, engine)
print(f"Total return ranking — {len(df_q1)} stocks")
df_q1

Total return ranking — 79 stocks


,ticker,company_name,market,sector,start_price,end_price,total_return_pct
0,NVDA,NVIDIA,US,Technology,0.42,137.28,32332.08
1,TSLA,Tesla,US,Consumer Disc.,1.59,417.41,26107.70
2,BAJFINANCE.NS,Bajaj Finance,India,Finance,3.03,684.60,22508.26
3,AVGO,Broadcom,US,Technology,1.33,232.98,17472.90
4,NFLX,Netflix,US,Communication,0.76,90.04,11685.73
...,...,...,...,...,...,...,...
74,XOM,Exxon Mobil,US,Energy,37.38,100.66,169.30
75,NESTLEIND.NS,Nestle India,India,FMCG,435.47,1062.85,144.07
76,ONGC.NS,ONGC,India,Energy,96.05,216.40,125.30
77,SBILIFE.NS,SBI Life Insurance,India,Finance,694.66,1399.24,101.43


In [4]:
## Q2: ANNUAL RETURN PER STOCK PER YEAR
q2 = """
WITH year_bounds AS (
    SELECT ticker, YEAR(trade_date) AS yr,
           MIN(trade_date) AS first_date, MAX(trade_date) AS last_date
    FROM daily_prices GROUP BY ticker, YEAR(trade_date)
)
SELECT yb.ticker, s.company_name, s.market, yb.yr AS year,
       ROUND(((lp.close_price - fp.close_price)/fp.close_price)*100, 2) AS annual_return_pct
FROM year_bounds yb
JOIN stocks s ON yb.ticker = s.ticker
JOIN daily_prices fp ON yb.ticker = fp.ticker AND fp.trade_date = yb.first_date
JOIN daily_prices lp ON yb.ticker = lp.ticker AND lp.trade_date = yb.last_date
ORDER BY yb.ticker, yb.yr
"""
df_q2 = pd.read_sql(q2, engine)
print(f"Annual returns — {df_q2['year'].nunique()} years × {df_q2['ticker'].nunique()} stocks")
df_q2.head(20)

Annual returns — 15 years × 79 stocks


,ticker,company_name,market,year,annual_return_pct
0,AAPL,Apple,US,2010,50.72
1,AAPL,Apple,US,2011,22.89
2,AAPL,Apple,US,2012,30.56
3,AAPL,Apple,US,2013,4.75
4,AAPL,Apple,US,2014,42.63
5,AAPL,Apple,US,2015,-2.08
6,AAPL,Apple,US,2016,12.38
7,AAPL,Apple,US,2017,48.04
8,AAPL,Apple,US,2018,-7.05
9,AAPL,Apple,US,2019,88.74


In [5]:
## Q3 & Q4: TOP 10 GAINERS AND LOSERS
base_cte = """
WITH first_date AS (SELECT ticker, MIN(trade_date) AS dt FROM daily_prices GROUP BY ticker),
     last_date  AS (SELECT ticker, MAX(trade_date) AS dt FROM daily_prices GROUP BY ticker)
SELECT s.ticker, s.company_name, s.market, s.sector,
       ROUND(((lp.close_price-fp.close_price)/fp.close_price)*100,2) AS total_return_pct
FROM stocks s
JOIN first_date fd ON s.ticker=fd.ticker
JOIN daily_prices fp ON s.ticker=fp.ticker AND fp.trade_date=fd.dt
JOIN last_date ld ON s.ticker=ld.ticker
JOIN daily_prices lp ON s.ticker=lp.ticker AND lp.trade_date=ld.dt
"""
gainers = pd.read_sql(base_cte + " ORDER BY total_return_pct DESC LIMIT 10", engine)
losers  = pd.read_sql(base_cte + " ORDER BY total_return_pct ASC  LIMIT 10", engine)

print("TOP 10 GAINERS")
display(gainers)
print("\nTOP 10 LOSERS")
display(losers)

TOP 10 GAINERS


,ticker,company_name,market,sector,total_return_pct
0,NVDA,NVIDIA,US,Technology,32332.08
1,TSLA,Tesla,US,Consumer Disc.,26107.70
2,BAJFINANCE.NS,Bajaj Finance,India,Finance,22508.26
3,AVGO,Broadcom,US,Technology,17472.90
4,NFLX,Netflix,US,Communication,11685.73
5,EICHERMOT.NS,Eicher Motors,India,Auto,7917.33
6,TITAN.NS,Titan Company,India,Infra/Other,4851.62
7,BAJAJFINSV.NS,Bajaj Finserv,India,Finance,4439.03
8,ADANIENT.NS,Adani Enterprises,India,Infra/Other,4402.00
9,AAPL,Apple,US,Technology,3811.64



TOP 10 LOSERS


,ticker,company_name,market,sector,total_return_pct
0,HDFCLIFE.NS,HDFC Life Insurance,India,Finance,82.86
1,SBILIFE.NS,SBI Life Insurance,India,Finance,101.43
2,ONGC.NS,ONGC,India,Energy,125.30
3,NESTLEIND.NS,Nestle India,India,FMCG,144.07
4,XOM,Exxon Mobil,US,Energy,169.30
5,NTPC.NS,NTPC,India,Energy,186.83
6,COALINDIA.NS,Coal India,India,Energy,205.55
7,CVX,Chevron,US,Energy,228.14
8,TATASTEEL.NS,Tata Steel,India,Metal,231.76
9,JNJ,Johnson & Johnson,US,Healthcare,244.11


In [6]:
## Q5: SECTOR-WISE PERFORMANCE
q5 = """
SELECT s.market, s.sector, COUNT(DISTINCT s.ticker) AS num_stocks,
       ROUND(AVG(sf.daily_return)*252*100,    2) AS annualised_return_pct,
       ROUND(STD(sf.daily_return)*SQRT(252)*100, 2) AS annualised_volatility_pct
FROM stock_features sf JOIN stocks s ON sf.ticker=s.ticker
WHERE sf.daily_return IS NOT NULL
GROUP BY s.market, s.sector
ORDER BY s.market, annualised_return_pct DESC
"""
df_q5 = pd.read_sql(q5, engine)
print("SECTOR PERFORMANCE BY MARKET")
df_q5

SECTOR PERFORMANCE BY MARKET


,market,sector,num_stocks,annualised_return_pct,annualised_volatility_pct
0,India,Finance,5,27.87,35.27
1,India,Infra/Other,9,23.79,33.87
2,India,Auto,5,22.01,29.11
3,India,Pharma,5,20.79,28.48
4,India,IT,5,20.02,27.28
5,India,FMCG,5,19.93,27.21
6,India,Banking,6,18.77,31.87
7,India,Metal,3,18.39,37.67
8,India,Energy,6,14.82,29.24
9,US,Communication,3,31.64,40.16


In [7]:
## Q6: INDIA vs US HEAD-TO-HEAD
q6 = """
SELECT market, COUNT(DISTINCT ticker) AS num_stocks,
       ROUND(AVG(daily_return)*252*100,              4) AS annualised_return_pct,
       ROUND(STD(daily_return)*SQRT(252)*100,        4) AS annualised_volatility_pct,
       ROUND(AVG(daily_return)/NULLIF(STD(daily_return),0)*SQRT(252), 4) AS sharpe_proxy
FROM stock_features WHERE daily_return IS NOT NULL GROUP BY market
"""
df_q6 = pd.read_sql(q6, engine)
print("INDIA vs US — HEAD-TO-HEAD")
df_q6

INDIA vs US — HEAD-TO-HEAD


,market,num_stocks,annualised_return_pct,annualised_volatility_pct,sharpe_proxy
0,India,49,20.75,31.12,0.67
1,US,30,23.25,31.35,0.74


In [11]:
## Q7: MONTHLY SEASONALITY + Q8: BULL/BEAR SIGNALS
import calendar
from sqlalchemy import text

# ── Q7: Monthly seasonality via SQL ──────────────────────────
q7 = """
SELECT market, MONTH(trade_date) AS month_num,
       ROUND(AVG(daily_return)*100, 4) AS avg_daily_return_pct
FROM stock_features WHERE daily_return IS NOT NULL
GROUP BY market, MONTH(trade_date)
ORDER BY market, month_num
"""
with engine.connect() as conn:
    r7 = conn.execute(text(q7))
    df_q7 = pd.DataFrame(r7.mappings().all())

df_q7['month'] = df_q7['month_num'].apply(lambda x: calendar.month_abbr[x])
print("MONTHLY SEASONALITY")
display(df_q7)

# ── Q8: Bull/Bear signals — done in pandas, no SQL subquery ──
df_sf = pd.read_sql(
    "SELECT ticker, trade_date, close_price, ma_200 FROM stock_features WHERE ma_200 IS NOT NULL",
    engine
)
df_st = pd.read_sql("SELECT ticker, company_name, market FROM stocks", engine)

# Get the most recent row per ticker
df_latest = df_sf.sort_values('trade_date').groupby('ticker').last().reset_index()

df_q8 = df_latest.merge(df_st, on='ticker')
df_q8['pct_above_ma200'] = ((df_q8['close_price'] - df_q8['ma_200']) / df_q8['ma_200'] * 100).round(2)
df_q8['signal']          = df_q8['close_price'].gt(df_q8['ma_200']).map({True:'Bull', False:'Bear'})
df_q8 = (df_q8[['ticker','company_name','market','close_price','ma_200','pct_above_ma200','signal']]
           .sort_values(['market','pct_above_ma200'], ascending=[True, False]))

print("\nBULL / BEAR SIGNALS")
display(df_q8)

MONTHLY SEASONALITY


,market,month_num,avg_daily_return_pct,month
0,India,1,0.0027,Jan
1,India,2,-0.0040,Feb
2,India,3,0.0757,Mar
3,India,4,0.1652,Apr
4,India,5,0.0793,May
...,...,...,...,...
19,US,8,0.0593,Aug
20,US,9,-0.0078,Sep
21,US,10,0.1150,Oct
22,US,11,0.2070,Nov



BULL / BEAR SIGNALS


,ticker,company_name,market,close_price,ma_200,pct_above_ma200,signal
26,DIVISLAB.NS,Divi's Laboratories,India,6017.60,4838.60,24.37,Bull
31,HCLTECH.NS,HCL Technologies,India,1810.27,1529.40,18.36,Bull
70,TECHM.NS,Tech Mahindra,India,1691.90,1439.69,17.52,Bull
76,WIPRO.NS,Wipro,India,284.69,244.06,16.65,Bull
48,M&M.NS,Mahindra & Mahindra,India,2978.97,2643.94,12.67,Bull
...,...,...,...,...,...,...,...
25,CVX,Chevron,US,134.18,141.91,-5.45,Bear
42,JNJ,Johnson & Johnson,US,137.56,146.36,-6.02,Bear
46,LLY,Eli Lilly,US,765.51,828.24,-7.57,Bear
78,XOM,Exxon Mobil,US,100.66,109.20,-7.82,Bear


In [ ]:
## Q9: TOP 10 HIGHEST VOLUME DAYS — INDIA
q9_india = """
SELECT dp.ticker, s.company_name, dp.trade_date,
       dp.volume, dp.close_price,
       ROUND(sf.daily_return*100, 2) AS daily_return_pct
FROM daily_prices dp
JOIN stocks s ON dp.ticker=s.ticker
JOIN stock_features sf ON dp.ticker=sf.ticker AND dp.trade_date=sf.trade_date
WHERE dp.market='India'
ORDER BY dp.volume DESC LIMIT 10
"""
q9_us = q9_india.replace("'India'", "'US'")

print("TOP VOLUME DAYS — INDIA"); display(pd.read_sql(q9_india, engine))
print("\nTOP VOLUME DAYS — US");   display(pd.read_sql(q9_us,    engine))

TOP VOLUME DAYS — INDIA


,ticker,company_name,trade_date,volume,close_price,daily_return_pct
0,TATASTEEL.NS,Tata Steel,2020-11-17,642845990,44.09,6.20
1,TATASTEEL.NS,Tata Steel,2021-02-10,573979190,58.22,-1.28
2,TATASTEEL.NS,Tata Steel,2021-05-07,540754330,99.73,7.40
3,TATASTEEL.NS,Tata Steel,2021-04-08,530630560,77.47,4.98
4,TATASTEEL.NS,Tata Steel,2017-05-17,518583508,36.39,8.18
5,POWERGRID.NS,Power Grid Corporation,2013-12-19,508752991,33.76,-0.75
6,COALINDIA.NS,Coal India,2010-11-04,479716245,113.88,NaN
7,TATASTEEL.NS,Tata Steel,2020-08-14,477423650,35.30,1.31
8,TATASTEEL.NS,Tata Steel,2021-05-06,464345370,92.86,2.87
9,TATASTEEL.NS,Tata Steel,2020-11-27,462912480,48.70,1.57



TOP VOLUME DAYS — US


,ticker,company_name,trade_date,volume,close_price,daily_return_pct
0,NVDA,NVIDIA,2017-06-09,3692928000,3.69,-6.47
1,NVDA,NVIDIA,2011-01-06,3493312000,0.44,13.84
2,NVDA,NVIDIA,2011-02-17,3470096000,0.59,9.85
3,NVDA,NVIDIA,2011-01-12,3431896000,0.53,14.95
4,NVDA,NVIDIA,2011-08-12,3195784000,0.29,-3.94
5,NFLX,Netflix,2011-10-25,3155418000,1.11,-34.89
6,NVDA,NVIDIA,2011-01-11,2711088000,0.47,-1.55
7,NVDA,NVIDIA,2011-01-13,2695192000,0.54,0.19
8,NVDA,NVIDIA,2010-07-29,2664476000,0.21,-9.87
9,NVDA,NVIDIA,2011-01-07,2579984000,0.45,2.80
